# Simple Vertex AI MLOps Demo
End-to-end demo using the Iris dataset.

Replace the values below before running.

In [ ]:
!pip -q install google-cloud-aiplatform scikit-learn joblib

In [22]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")


Authenticated in Colab.


In [ ]:
!gsutil mb -p himanshugcpproject -l us-central1 gs://himanshugcpproject-mlops-demo-2026

In [23]:
!gsutil ls -p himanshugcpproject

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://cloud-ai-platform-b4fb718e-d149-4658-a431-248964cfbb6b/
gs://gcf-v2-sources-406774515547-us-central1/
gs://gcf-v2-uploads-406774515547.us-central1.cloudfunctions.appspot.com/
gs://himanshugcpproject-mlops-demo-2026/
gs://himanshugcpproject-mlops-lab-1783921688/
gs://himanshugcpproject-mlops-lab-1783921814/
gs://himanshugcpproject-mlops-lab-1783921971/
gs://himanshugcpproject-mlops-lab-1783922579/
gs://himanshugcpproject-mlops-lab-1783923265/
gs://himanshugcpproject-mlops-lab-1783923611/
gs://himanshugcpproject-mlops-lab-1783924036/
gs://himanshugcpproject-mlops-lab-1783940418/
gs://himanshugcpproject-mlops-lab-1783940791/
gs://himanshugcpproject-mlops-lab-1783942709/
gs://himanshugcpproject_cloudbuild/
gs://himanshugcpprojectr

In [24]:
PROJECT_ID = "himanshugcpproject"
REGION = "us-central1"
BUCKET_URI = "gs://himanshugcpproject-mlops-demo-2026"

from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)


## Train a model

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib, os

iris=load_iris()
X_train,X_test,y_train,y_test=train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

model=RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train,y_train)

pred=model.predict(X_test)
print("Accuracy:", accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

os.makedirs("model", exist_ok=True)
joblib.dump(model,"model/model.joblib")


## Upload model to Vertex AI Model Registry

In [ ]:
MODEL = aiplatform.Model.upload(
    display_name="iris-rf-demo",
    artifact_uri="model",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
)
MODEL.wait()
print(MODEL.resource_name)


## Create endpoint and deploy

In [20]:
endpoint = aiplatform.Endpoint.create(display_name="iris-endpoint")
endpoint.deploy(model=MODEL, machine_type="n1-standard-2")
print(endpoint.resource_name)


projects/406774515547/locations/us-central1/endpoints/8952725600410271744


## Online prediction

In [25]:
sample = X_test[:3].tolist()
result = endpoint.predict(instances=sample)
print(result.predictions)


[1.0, 0.0, 2.0]


## Train Version 2

In [26]:
model_v2=RandomForestClassifier(n_estimators=300, random_state=7)
model_v2.fit(X_train,y_train)
os.makedirs("model_v2", exist_ok=True)
joblib.dump(model_v2,"model_v2/model.joblib")

MODEL_V2=aiplatform.Model.upload(
    display_name="iris-rf-demo",
    artifact_uri="model_v2",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
)
MODEL_V2.wait()
print(MODEL_V2.resource_name)


projects/406774515547/locations/us-central1/models/2827575020488753152


## Deploy Version 2 with 10% traffic

In [27]:
endpoint.deploy(
    model=MODEL_V2,
    machine_type="n1-standard-2",
    traffic_percentage=10,
)
print("Version 2 deployed with 10% traffic.")


Version 2 deployed with 10% traffic.


## Cleanup

In [30]:
# Uncomment when finished
# endpoint.undeploy_all()
# endpoint.delete(force=True)
MODEL.delete()
MODEL_V2.delete()
